# Fehleranalyse

**Autor:** jd · **Datum:** 2026-10-12

**Ziel:** Erklären, welche Klassen verwechselt werden und warum Baseline, kombinierter und hierarchischer Ansatz unterschiedlich abschneiden (#32).

In [ ]:
%load_ext autoreload
%autoreload 2

import itertools

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

from awp2.config import (
    AEZ_COL,
    CLOSED_CANOPY_STAGES,
    CROP_COL,
    EARLY_STAGES,
    MODEL_DOCS_FIGURES_DIR,
    MONTH_COL,
    RANDOM_FOREST_N_ESTIMATORS,
    RECALL_COL,
    SEED,
    STAGE_COL,
    SUPPORT_COL,
)
from awp2.data import band_columns, load_split
from awp2.evaluation import compare_predictions, error_breakdown, recall_by_class
from awp2.experiment import RunConfig, out_of_fold_predictions, run
from awp2.models import make_combined_svm, make_hierarchical_svm, oracle_crop_metrics
from awp2.plots import (
    plot_error_breakdown,
    plot_error_rate_by,
    plot_metric_comparison,
    plot_recall_comparison,
    plot_spectra,
    save_doc_figure,
)
from awp2.preprocessing import PreprocessingConfig, ndvi

In [ ]:
split = load_split()
X_val, y_val = split.X_val, split.y_val
X_val.shape

## 1. Die drei Ansätze nachbauen

Je Ansatz das beste bewertete Modell mit seinen protokollierten Einstellungen, neu trainiert auf
dem Trainingsteil und auf der gemeinsamen Validierung vorhergesagt (`track=False`, keine neuen
Läufe). Es wird nichts mehr eingestellt – die Validierung dient hier nur der Analyse.

| Ansatz | Lauf | Einstellungen |
| --- | --- | --- |
| Baseline | `rf_baseline` | Random Forest, `max_depth=20`, `max_features=0.3`, numerischer Monat |
| Kombiniert | `combined_svm_expanded` | RBF-SVM, `C=100`, `gamma=0.003`, `scale=True`, numerischer Monat |
| Hierarchisch | `hierarchical_svm` | RBF-SVM → RBF-SVM, Kultur `C=30`/`gamma=0.01`, Stadium `C=100`/`gamma=0.003` |

`rf_baseline` lässt sich nicht exakt reproduzieren: Das gespeicherte M1-Modell ist mit der
heutigen Pipeline nicht mehr ladbar (Interpolation geändert in `585fbf2`); nachtrainiert ergibt
sich BAcc kombiniert 0.750 statt 0.743.

In [ ]:
approaches = {
    "Baseline (RF)": (
        "baseline",
        RandomForestClassifier(
            n_estimators=RANDOM_FOREST_N_ESTIMATORS,
            max_depth=20,
            max_features=0.3,
            min_samples_leaf=1,
            class_weight="balanced",
            random_state=SEED,
            n_jobs=-1,
        ),
        PreprocessingConfig(use_cyclic_month=False),
    ),
    "Kombiniert (SVM)": (
        "combined",
        make_combined_svm().set_params(estimator__C=100.0, estimator__gamma=0.003),
        PreprocessingConfig(scale=True, use_cyclic_month=False),
    ),
    "Hierarchisch (SVM)": (
        "hierarchical",
        make_hierarchical_svm().set_params(
            crop_estimator__C=30.0,
            crop_estimator__gamma=0.01,
            stage_estimator__C=100.0,
            stage_estimator__gamma=0.003,
        ),
        PreprocessingConfig(use_spectral_standard_scale=True),
    ),
}
results = {
    label: run(
        model,
        RunConfig(
            name="error_analysis_refit",
            approach=approach,
            preprocessing=preprocessing,
            track=False,
            log_model=False,
        ),
    )
    for label, (approach, model, preprocessing) in approaches.items()
}
predictions = {label: result.y_pred for label, result in results.items()}
pd.DataFrame({label: result.metrics.model_dump() for label, result in results.items()}).round(3)

In [ ]:
metrics_ax = plot_metric_comparison({label: r.metrics for label, r in results.items()})
metrics_ax.set_title("Die drei Ansätze auf der Validierung");

## 2. Wo entstehen die Fehler?

Jede Zeile ist entweder ganz richtig, hat die falsche Kultur oder nur das falsche Stadium. Beim
hierarchischen Ansatz zeigt die Oracle-Kultur, was ohne Kulturfehler möglich wäre.

In [ ]:
breakdowns = {label: error_breakdown(y_val, pred) for label, pred in predictions.items()}
breakdown_ax = plot_error_breakdown(breakdowns)
breakdown_ax.set_title("Fehlerzerlegung auf der Validierung")
save_doc_figure(breakdown_ax.figure, "fehleranalyse_zerlegung", MODEL_DOCS_FIGURES_DIR)
pd.DataFrame({label: b.model_dump() for label, b in breakdowns.items()}).round(3)

In [ ]:
oracle = oracle_crop_metrics(results["Hierarchisch (SVM)"].pipeline, X_val, y_val)
oracle.bacc_combined, oracle.bacc_stage

## 3. Unterscheiden sich die Ansätze wirklich?

McNemar-Test je Paar auf kombinierte Treffer (nicht BAcc, siehe `compare_predictions`) und die
Überschneidung der Kulturfehler: Machen die Modelle dieselben Fehler?

In [ ]:
pd.DataFrame(
    {
        f"{first} vs. {second}": compare_predictions(
            y_val, predictions[first], predictions[second]
        ).model_dump()
        for first, second in itertools.combinations(predictions, 2)
    }
).T

In [ ]:
crop_wrong = {label: pred[CROP_COL] != y_val[CROP_COL] for label, pred in predictions.items()}
pd.Series(
    {
        "Kombiniert": crop_wrong["Kombiniert (SVM)"].sum(),
        "Hierarchisch": crop_wrong["Hierarchisch (SVM)"].sum(),
        "beide SVMs": (crop_wrong["Kombiniert (SVM)"] & crop_wrong["Hierarchisch (SVM)"]).sum(),
        "Baseline": crop_wrong["Baseline (RF)"].sum(),
        "alle drei": pd.concat(crop_wrong, axis=1).all(axis=1).sum(),
    },
    name="Zeilen mit falscher Kultur",
)

## 4. Welche Klassen gewinnt oder verliert ein Ansatz?

Die Balanced Accuracy ist der Mittelwert der Recalls je Klasse – kleine Klassen zählen so viel
wie große. Hier die Paare mit den größten Unterschieden.

In [ ]:
recalls = {label: recall_by_class(y_val, pred) for label, pred in predictions.items()}
recall_figure = plot_recall_comparison(recalls, top=12)
save_doc_figure(recall_figure, "fehleranalyse_recall", MODEL_DOCS_FIGURES_DIR)
recall_table = pd.DataFrame({label: r[RECALL_COL] for label, r in recalls.items()})
recall_table[SUPPORT_COL] = recalls["Kombiniert (SVM)"][SUPPORT_COL]
recall_table["hierarchisch − kombiniert"] = (
    recall_table["Hierarchisch (SVM)"] - recall_table["Kombiniert (SVM)"]
)
recall_table.sort_values("hierarchisch − kombiniert").round(3)

## 5. Wann und wo liegt die Kultur falsch?

Kulturfehlerrate nach Monat und AEZ. Fehler, die sich auf bestimmte Monate oder Regionen
konzentrieren, sprechen für Spektren, die sich dort zu ähnlich sind – etwa vor dem Auflaufen oder
nach der Ernte.

In [ ]:
month_ax = plot_error_rate_by(crop_wrong, X_val[MONTH_COL], xlabel="Monat")
month_ax.set_title("Kulturfehler nach Monat")
save_doc_figure(month_ax.figure, "fehleranalyse_monat", MODEL_DOCS_FIGURES_DIR)

In [ ]:
aez_ax = plot_error_rate_by(crop_wrong, X_val[AEZ_COL], xlabel="AEZ")
aez_ax.set_title("Kulturfehler nach AEZ")
save_doc_figure(aez_ax.figure, "fehleranalyse_aez", MODEL_DOCS_FIGURES_DIR)

Gegenprobe ohne Validierung: dasselbe Muster für das hierarchische Modell auf
Out-of-fold-Vorhersagen des Trainingsteils. Nur Muster, die auch hier auftreten, taugen als
Grundlage für neue Hypothesen – deren Nutzen entscheidet dann wieder die CV.

In [ ]:
hierarchical_model, hierarchical_preprocessing = approaches["Hierarchisch (SVM)"][1:]
oof = out_of_fold_predictions(
    hierarchical_model,
    RunConfig(
        name="error_analysis_oof",
        approach="hierarchical",
        preprocessing=hierarchical_preprocessing,
        track=False,
    ),
)
oof_crop_wrong = oof[CROP_COL] != split.y_train[CROP_COL]
(
    pd.DataFrame(
        {
            "Validierung": crop_wrong["Hierarchisch (SVM)"].groupby(X_val[MONTH_COL]).mean(),
            "Training (out-of-fold)": oof_crop_wrong.groupby(split.X_train[MONTH_COL]).mean(),
        }
    ).round(3),
    pd.DataFrame(
        {
            "Validierung": crop_wrong["Hierarchisch (SVM)"].groupby(X_val[AEZ_COL]).mean(),
            "Training (out-of-fold)": oof_crop_wrong.groupby(split.X_train[AEZ_COL]).mean(),
            "Zeilen Validierung": X_val[AEZ_COL].value_counts(),
        }
    ).round(3),
)

## 6. Die häufigsten Verwechslungen spektral erklärt

Für das hierarchische Modell: Welche Stadien, Monate und Regionen stecken hinter den häufigsten
Kulturverwechslungen, und wie sehen ihre Spektren aus?

In [ ]:
hierarchical = predictions["Hierarchisch (SVM)"]
confusions = (
    pd.DataFrame({"wahr": y_val[CROP_COL], "vorhergesagt": hierarchical[CROP_COL]})
    .loc[crop_wrong["Hierarchisch (SVM)"]]
    .value_counts()
    .head(5)
)
confusions

In [ ]:
def describe(mask: pd.Series) -> dict[str, dict]:
    return {
        "wahres Stadium": y_val.loc[mask, STAGE_COL].value_counts().to_dict(),
        "Monat": X_val.loc[mask, MONTH_COL].value_counts().sort_index().to_dict(),
        "AEZ": X_val.loc[mask, AEZ_COL].value_counts().to_dict(),
    }


def mistaken_as(true_crop: str, predicted_crop: str) -> pd.Series:
    return (y_val[CROP_COL] == true_crop) & (hierarchical[CROP_COL] == predicted_crop)


top_true, top_predicted = confusions.index[0]
top_confusion = mistaken_as(top_true, top_predicted)
describe(top_confusion)

### Häufigste Verwechslung: Spektren und NDVI

Verglichen mit den richtig erkannten Zeilen derselben Kultur und der verwechselten Kultur in
denselben AEZ.

In [ ]:
same_aez = X_val[AEZ_COL].isin(X_val.loc[top_confusion, AEZ_COL].unique())
groups = {
    f"{top_true}, als {top_predicted} erkannt": top_confusion,
    f"{top_true}, richtig erkannt (gleiche AEZ)": (y_val[CROP_COL] == top_true)
    & ~crop_wrong["Hierarchisch (SVM)"]
    & same_aez,
    f"{top_predicted} (gleiche AEZ)": (y_val[CROP_COL] == top_predicted) & same_aez,
}
top_frame = pd.concat([X_val.loc[mask].assign(Gruppe=label) for label, mask in groups.items()])
top_ax = plot_spectra(top_frame, by="Gruppe", title=f"{top_true} → {top_predicted}")
save_doc_figure(top_ax.figure, "fehleranalyse_spektren_weizen", MODEL_DOCS_FIGURES_DIR)
bands = band_columns(X_val)
pd.DataFrame(
    {
        "Zeilen": top_frame.groupby("Gruppe").size(),
        "NDVI Median": top_frame.groupby("Gruppe").apply(lambda g: ndvi(g).median()),
        "mittlere Reflektanz": top_frame.groupby("Gruppe")[bands].mean().mean(axis=1),
        "Stadien wie die Verwechselten": top_frame.groupby("Gruppe").apply(
            lambda g: (
                y_val.loc[g.index, STAGE_COL]
                .isin(y_val.loc[top_confusion, STAGE_COL].unique())
                .sum()
            )
        ),
    }
).round(2)

### Das Paar mit gegenseitiger Verwechslung

Das zweit- und dritthäufigste Paar sind dieselben zwei Kulturen in beide Richtungen. Getrennt nach
Stadiengruppe: jede falsche Kultur und speziell die Verwechslung mit der anderen Kultur des Paars.

In [ ]:
pair = confusions.index[1]
stage_groups = {"früh": list(EARLY_STAGES), "geschlossener Bestand": list(CLOSED_CANOPY_STAGES)}
pair_rates = pd.DataFrame(
    {
        f"{crop}, {stage_label}": {
            "Zeilen": mask.sum(),
            "Kultur falsch (beliebig)": crop_wrong["Hierarchisch (SVM)"][mask].mean(),
            "als andere Kultur des Paars": (hierarchical.loc[mask, CROP_COL] == other).mean(),
        }
        for crop, other in [pair, pair[::-1]]
        for stage_label, stages in stage_groups.items()
        for mask in [(y_val[CROP_COL] == crop) & y_val[STAGE_COL].isin(stages)]
    }
).T
pair_rates.round(3)

In [ ]:
pair_figure, pair_axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
for ax, (stage_label, stages) in zip(pair_axes, stage_groups.items(), strict=True):
    in_stages = y_val[STAGE_COL].isin(stages)
    pair_frame = pd.concat(
        [X_val.loc[in_stages & (y_val[CROP_COL] == crop)].assign(Gruppe=crop) for crop in pair]
    )
    plot_spectra(pair_frame, by="Gruppe", ax=ax, title=f"{pair[0]} vs. {pair[1]}: {stage_label}")
pair_figure.tight_layout()
save_doc_figure(pair_figure, "fehleranalyse_spektren_mais_soja", MODEL_DOCS_FIGURES_DIR)